# MaxSig Stage-Placement Ablation Study

This notebook runs a controlled ablation over **which encoder stage(s)** `MaxSigLayerNormalized`
is inserted into, using the same U-Net / dataset / training loop as `UNet-maxsig.py`.

For each scenario the notebook trains one model and reports only:
- **Best validation Dice** (and the epoch it occurred at)
- **Test Dice** (evaluated with the corresponding best checkpoint)

For the **default configuration** (`maxsig_down_stages=(1,2)`) it additionally reports:
- **GFLOPs** (at 256x256 input)
- **FPS** (inference throughput)

Encoder stage <-> channel width map (for `features=(64,128,256,512)`):

| stage index | channels |
|---|---|
| 0 | 64  |
| 1 | 128 |
| 2 | 256 |
| 3 | 512 |

Scenarios covered (includes the reviewer-requested **Stage 0 (64ch)** test):

1. Baseline — no MaxSig anywhere: `()`
2. MaxSig at **Stage 0 only (64ch)** — reviewer's specific question
3. MaxSig at Stage 1 only (128ch)
4. MaxSig at Stage 2 only (256ch)
5. MaxSig at Stage 3 only (512ch)
6. **Default**: Stages (1,2) — 128ch + 256ch
7. MaxSig at all stages (0,1,2,3)

Each scenario is a self-contained cell so you can re-run / skip individual scenarios independently.


In [1]:
# ============================================================
# Config
# ============================================================
ROOT = '/home/jupyter-r.yazdi/myproject/dti3d/0pycodes/UnetMaxsig/compsitdataset/'

config = {
    "lr": 1e-4,
    "batch_size": 8,
    "images_dir": ROOT,
    "masks_dir":  ROOT,
    "epochs": 50,          # epochs used for the *default* training run in the original script
}

# Ablation-specific settings. Feel free to lower this for a faster sweep
# (e.g. 40-60 epochs) — every scenario cell reads from here.
ABLATION_EPOCHS = config['epochs']

RESULTS_DIR = ROOT + 'ablation_ckpts/'


In [2]:
import os
os.makedirs(RESULTS_DIR, exist_ok=True)

from time import perf_counter
import numpy as np
import cv2
import matplotlib.pyplot as plt
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
from torchvision.utils import make_grid
import torchvision.transforms.functional as TF

from torch.optim.lr_scheduler import StepLR
from torch.utils.data import DataLoader, Dataset
from glob import glob
import albumentations as A
from albumentations.pytorch import ToTensorV2

torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)


device: cuda


/home/jupyter-r.yazdi/.local/lib/python3.10/site-packages/albumentations/check_version.py:49: UserWarning: Error fetching version info <urlopen error [Errno 101] Network is unreachable>
  data = fetch_version_info()


## Dataset & Dataloaders\n(unchanged from `UNet-maxsig.py`)

In [3]:
class mydataset(Dataset):
    def __init__(self, images_dir, masks_dir, transforms_img=None):
        self.images_dir = images_dir
        self.masks_dir = masks_dir
        self.transforms_img = transforms_img
        self.mask_names = glob(masks_dir + "/*.tif") + glob(masks_dir + "/*.bmp") + glob(masks_dir + "/*.png")

    def __len__(self):
        return len(self.mask_names)

    def __getitem__(self, index):
        mask_name = self.mask_names[index]
        mask = cv2.imread(mask_name, cv2.IMREAD_ANYDEPTH)
        mask = (mask - mask.min()) / (mask.max() - mask.min())
        mask = (mask > 0).astype(np.float32)
        mask = cv2.resize(mask, (256, 256), interpolation=cv2.INTER_AREA)

        imgn = mask_name.replace('/masks/', '/images/')
        if imgn.endswith('.tif'):
            if not os.path.exists(imgn):
                imgn = imgn.replace('.tif', '.bmp')

        image = cv2.imread(imgn, cv2.IMREAD_ANYDEPTH)
        image = (image - image.min()) / (image.max() - image.min()).astype(np.float32)
        image = cv2.resize(image, (256, 256), interpolation=cv2.INTER_AREA)

        if self.transforms_img is not None:
            transformed = self.transforms_img(image=image, mask=mask)
            image = transformed["image"]
            mask = transformed["mask"]

        return image, mask


In [4]:
batch_size = config['batch_size']

train_transform = A.Compose([A.Resize(256, 256), ToTensorV2()])
val_transform   = A.Compose([A.Resize(256, 256), ToTensorV2()])
test_transform  = A.Compose([A.Resize(256, 256), ToTensorV2()])

train_ds = mydataset(ROOT + 'dataset/train/images/', ROOT + 'dataset/train/masks/', train_transform)
val_ds   = mydataset(ROOT + 'dataset/val/images/',   ROOT + 'dataset/val/masks/',   val_transform)
test_ds  = mydataset(ROOT + 'dataset/test/images/',  ROOT + 'dataset/test/masks/',  test_transform)

train_dataset = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
val_dataset   = DataLoader(val_ds,   batch_size=batch_size, shuffle=True)
test_dataset  = DataLoader(test_ds,  batch_size=batch_size, shuffle=True)

print("train batches:", len(train_dataset), "| val batches:", len(val_dataset), "| test batches:", len(test_dataset))


train batches: 37 | val batches: 8 | test batches: 8


## Model definitions\n(unchanged from `UNet-maxsig.py`: `MaxSigLayerNormalized`, `DoubleConv`, `DownBlockWithMaxSig`, `UNet`)

In [3]:
class MaxSigLayerNormalized(nn.Module):
    def __init__(self, ch_in: int, ch_out: int, kernel_size: int, stride: int = 1, use_sigmoid_input: bool = True):
        super().__init__()
        assert kernel_size >= 1
        self.ch_in = ch_in
        self.ch_out = ch_out
        self.kernel_size = kernel_size
        self.stride = stride
        self.use_sigmoid_input = use_sigmoid_input

        self.padding = (kernel_size - 1) // 2
        self.kernel_area = kernel_size ** 2
        self.center_index = self.kernel_area // 2

        self.weight = nn.Parameter(torch.rand(1, 1, self.kernel_area, 1, 1))
        self.bias   = nn.Parameter(torch.zeros(1, 1, self.kernel_area, 1, 1))

        self.weight_center = nn.Parameter(torch.tensor(1.0))
        self.weight_median = nn.Parameter(torch.tensor(1.0))

        self.proj = nn.Conv2d(ch_in, ch_out, kernel_size=1, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, C_in, H, W = x.shape
        assert C_in == self.ch_in, f"Expected {self.ch_in} channels, got {C_in}"

        x_in = torch.sigmoid(x) if self.use_sigmoid_input else x

        x_unfold = F.unfold(x_in, kernel_size=self.kernel_size, padding=self.padding, stride=self.stride)
        _, CK2, L = x_unfold.shape
        assert CK2 == C_in * self.kernel_area

        out_h = (H + 2 * self.padding - self.kernel_size) // self.stride + 1
        out_w = (W + 2 * self.padding - self.kernel_size) // self.stride + 1

        x_unfold = x_unfold.view(B, C_in, self.kernel_area, out_h, out_w)

        w = torch.sigmoid(self.weight)
        b = torch.sigmoid(self.bias)

        D = torch.maximum(w, x_unfold) + b

        mean_stat = torch.mean(D, dim=2)
        median_stat, _ = torch.median(D, dim=2)

        center_values = x_unfold[:, :, self.center_index, :, :]

        out = (self.weight_center * center_values + self.weight_median * median_stat - mean_stat)
        out = self.proj(out)
        return out


class DoubleConv(nn.Module):
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, 1, 1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, 1, 1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.conv(x)


class DownBlockWithMaxSig(nn.Module):
    def __init__(self, in_ch, out_ch, use_maxsig, kernel_sig=3, maxsig_mode="gated",
                 gamma_init=0.0, use_sigmoid_input=True):
        super().__init__()
        assert maxsig_mode in ("gated", "direct")
        self.use_maxsig = use_maxsig
        self.maxsig_mode = maxsig_mode

        self.conv = DoubleConv(in_ch, out_ch)

        if use_maxsig:
            self.maxsig = MaxSigLayerNormalized(
                ch_in=out_ch, ch_out=out_ch, kernel_size=kernel_sig,
                stride=1, use_sigmoid_input=use_sigmoid_input,
            )
            self.post = nn.Sequential(nn.BatchNorm2d(out_ch), nn.LeakyReLU(0.1, inplace=True))
            if maxsig_mode == "gated":
                self.gamma = nn.Parameter(torch.tensor(gamma_init))
        else:
            self.maxsig = None
            self.post = None

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.conv(x)
        if not self.use_maxsig:
            return x
        m = self.maxsig(x)
        m = self.post(m)
        if self.maxsig_mode == "direct":
            return m
        return x + self.gamma * m


class UNet(nn.Module):
    def __init__(self, in_channels=1, out_channels=1, features=(64, 128, 256, 512),
                 maxsig_down_stages=(1, 2), maxsig_kernel: int = 3, maxsig_mode: str = "gated",
                 gamma_init: float = 0.0, use_sigmoid_input: bool = True):
        super().__init__()
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.downs = nn.ModuleList()
        self.ups = nn.ModuleList()

        maxsig_down_stages = set(maxsig_down_stages)
        ch = in_channels
        feats = list(features)

        for i, feat in enumerate(feats):
            use_maxsig = i in maxsig_down_stages
            self.downs.append(
                DownBlockWithMaxSig(
                    in_ch=ch, out_ch=feat, use_maxsig=use_maxsig, kernel_sig=maxsig_kernel,
                    maxsig_mode=maxsig_mode, gamma_init=gamma_init, use_sigmoid_input=use_sigmoid_input,
                )
            )
            ch = feat

        self.bottleneck = DoubleConv(feats[-1], feats[-1] * 2)

        for feat in reversed(feats):
            self.ups.append(nn.ConvTranspose2d(feat * 2, feat, kernel_size=2, stride=2))
            self.ups.append(DoubleConv(feat * 2, feat))

        self.final_conv = nn.Conv2d(feats[0], out_channels, kernel_size=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        skip_connections = []
        for down in self.downs:
            x = down(x)
            skip_connections.append(x)
            x = self.pool(x)

        x = self.bottleneck(x)
        skip_connections = skip_connections[::-1]

        for idx in range(0, len(self.ups), 2):
            x = self.ups[idx](x)
            skip = skip_connections[idx // 2]
            if x.shape[2:] != skip.shape[2:]:
                x = TF.resize(x, size=skip.shape[2:])
            x = torch.cat((skip, x), dim=1)
            x = self.ups[idx + 1](x)

        return self.final_conv(x)


## Ablation helpers

`run_scenario(...)` trains one model for a given `maxsig_down_stages` tuple, tracks the best
validation Dice / epoch, reloads that checkpoint, evaluates test Dice, and returns a single
results dict (nothing else is printed/plotted, to keep the ablation output compact).

`compute_flops_fps(...)` reports GFLOPs (via `thop`) and FPS (measured throughput) for a model.


In [4]:
def dice_from_logits(preds_logits, labels):
    pred_sig = torch.sigmoid(preds_logits)
    preds = (pred_sig > 0.5).float()
    intersection = (preds * labels).sum()
    union = (preds + labels).sum()
    return (2 * intersection / union)


def run_scenario(run_name, maxsig_down_stages, epochs=ABLATION_EPOCHS, verbose_every=10,
                  maxsig_kernel=3, maxsig_mode="gated", gamma_init=0.0, use_sigmoid_input=True):
    """Train + evaluate one MaxSig stage-placement configuration.

    Returns dict with: run_name, maxsig_down_stages, best_val_dice, best_epoch, test_dice
    """
    torch.manual_seed(42)

    model = UNet(
        in_channels=1, out_channels=1, features=(64, 128, 256, 512),
        maxsig_down_stages=maxsig_down_stages, maxsig_kernel=maxsig_kernel,
        maxsig_mode=maxsig_mode, gamma_init=gamma_init, use_sigmoid_input=use_sigmoid_input,
    ).to(device)

    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.AdamW(model.parameters(), lr=config['lr'])
    scheduler = StepLR(optimizer, step_size=100, gamma=0.83)
    scaler = torch.amp.GradScaler('cuda')

    ckpt_path = os.path.join(RESULTS_DIR, f"best-{run_name}.pt")
    best_valid_dice = 0.0
    best_epoch = 0

    for epoch in range(epochs):
        model.train()
        for inputs, mask in train_dataset:
            optimizer.zero_grad()
            with torch.autocast(device_type="cuda"):
                outputs = model(inputs.to(device))
                labels = mask.to(device).unsqueeze(1)
                loss = criterion(outputs, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        scheduler.step()

        dice_score = 0.0
        with torch.no_grad():
            model.eval()
            for inputs, mask in val_dataset:
                inputs = inputs.to(device)
                preds = model(inputs)
                labels = mask.to(device).unsqueeze(1)
                dice_score += dice_from_logits(preds, labels)

        epoch_dc = float(dice_score / len(val_dataset))
        if best_valid_dice < epoch_dc:
            best_valid_dice = epoch_dc
            best_epoch = epoch
            torch.save(model.state_dict(), ckpt_path)

        if verbose_every and (epoch + 1) % verbose_every == 0:
            print(f"[{run_name}] epoch {epoch+1}/{epochs} | val_dice={epoch_dc:.4f} "
                  f"| best={best_valid_dice:.4f}@{best_epoch+1}")

    # ---- test evaluation with the best checkpoint ----
    model.load_state_dict(torch.load(ckpt_path))
    dice_score = 0.0
    with torch.no_grad():
        model.eval()
        for inputs, mask in test_dataset:
            inputs = inputs.to(device)
            preds = model(inputs)
            labels = mask.to(device).unsqueeze(1)
            dice_score += dice_from_logits(preds, labels)
    test_dice = float(dice_score / len(test_dataset))

    result = {
        "run_name": run_name,
        "maxsig_down_stages": tuple(sorted(maxsig_down_stages)),
        "best_val_dice": best_valid_dice,
        "best_epoch": best_epoch + 1,
        "test_dice": test_dice,
    }
    print(f"[{run_name}] DONE | best_val_dice={best_valid_dice:.4f} (epoch {best_epoch+1}) "
          f"| test_dice={test_dice:.4f}")
    return result


In [5]:
def compute_flops_fps(model, input_size=(1, 1, 256, 256), n_warmup=10, n_iters=50):
    """Reports GFLOPs (thop) and FPS (measured wall-clock throughput) for a model."""
    model.eval()
    dummy = torch.randn(*input_size).to(device)

    # ---- GFLOPs ----
    try:
        from thop import profile
        macs, params = profile(model, inputs=(dummy,), verbose=False)
        gflops = macs * 2 / 1e9  # MACs -> FLOPs
    except ImportError:
        print("`thop` not installed. Run `pip install thop` to get GFLOPs. Skipping GFLOPs.")
        gflops = None

    # ---- FPS ----
    with torch.no_grad():
        for _ in range(n_warmup):
            _ = model(dummy)
        if device == "cuda":
            torch.cuda.synchronize()
        t0 = perf_counter()
        for _ in range(n_iters):
            _ = model(dummy)
        if device == "cuda":
            torch.cuda.synchronize()
        t1 = perf_counter()

    fps = n_iters / (t1 - t0)
    return gflops, fps


In [8]:
# Collect every scenario's result dict here for the final summary table.
ablation_results = []


### Scenario 1 — Baseline (no MaxSig)

In [9]:
res = run_scenario(run_name="baseline_no_maxsig", maxsig_down_stages=())
ablation_results.append(res)


/tmp/ipykernel_4161971/1748191668.py:14: RuntimeWarning: invalid value encountered in divide
  mask = (mask - mask.min()) / (mask.max() - mask.min())


[baseline_no_maxsig] epoch 10/50 | val_dice=0.6379 | best=0.6379@10
[baseline_no_maxsig] epoch 20/50 | val_dice=0.6166 | best=0.7268@18
[baseline_no_maxsig] epoch 30/50 | val_dice=0.6583 | best=0.7268@18
[baseline_no_maxsig] epoch 40/50 | val_dice=0.7032 | best=0.7525@38
[baseline_no_maxsig] epoch 50/50 | val_dice=0.7484 | best=0.7525@38
[baseline_no_maxsig] DONE | best_val_dice=0.7525 (epoch 38) | test_dice=0.6982


### Scenario 2 — MaxSig at **Stage 0 only (64 channels)**
> This is the configuration requested by the reviewer: *"Did you test applying MaxSig at stage 0 (64 channels) and find it hurt performance?"*


In [10]:
res = run_scenario(run_name="stage0_only_64ch", maxsig_down_stages=(0,))
ablation_results.append(res)


/tmp/ipykernel_4161971/1748191668.py:14: RuntimeWarning: invalid value encountered in divide
  mask = (mask - mask.min()) / (mask.max() - mask.min())


[stage0_only_64ch] epoch 10/50 | val_dice=0.7311 | best=0.7311@10
[stage0_only_64ch] epoch 20/50 | val_dice=0.6551 | best=0.7311@10
[stage0_only_64ch] epoch 30/50 | val_dice=0.7053 | best=0.7333@22
[stage0_only_64ch] epoch 40/50 | val_dice=0.7213 | best=0.7468@31
[stage0_only_64ch] epoch 50/50 | val_dice=0.7177 | best=0.7468@31
[stage0_only_64ch] DONE | best_val_dice=0.7468 (epoch 31) | test_dice=0.7231


### Scenario 3 — MaxSig at Stage 1 only (128 channels)

In [11]:
res = run_scenario(run_name="stage1_only_128ch", maxsig_down_stages=(1,))
ablation_results.append(res)


/tmp/ipykernel_4161971/1748191668.py:14: RuntimeWarning: invalid value encountered in divide
  mask = (mask - mask.min()) / (mask.max() - mask.min())


[stage1_only_128ch] epoch 10/50 | val_dice=0.5906 | best=0.6582@9
[stage1_only_128ch] epoch 20/50 | val_dice=0.6760 | best=0.7025@17
[stage1_only_128ch] epoch 30/50 | val_dice=0.6794 | best=0.7225@25
[stage1_only_128ch] epoch 40/50 | val_dice=0.6858 | best=0.7539@34
[stage1_only_128ch] epoch 50/50 | val_dice=0.6867 | best=0.7539@34
[stage1_only_128ch] DONE | best_val_dice=0.7539 (epoch 34) | test_dice=0.7330


### Scenario 4 — MaxSig at Stage 2 only (256 channels)

In [12]:
res = run_scenario(run_name="stage2_only_256ch", maxsig_down_stages=(2,))
ablation_results.append(res)


/tmp/ipykernel_4161971/1748191668.py:14: RuntimeWarning: invalid value encountered in divide
  mask = (mask - mask.min()) / (mask.max() - mask.min())


[stage2_only_256ch] epoch 10/50 | val_dice=0.5758 | best=0.6447@9
[stage2_only_256ch] epoch 20/50 | val_dice=0.5990 | best=0.7094@16
[stage2_only_256ch] epoch 30/50 | val_dice=0.7271 | best=0.7327@26
[stage2_only_256ch] epoch 40/50 | val_dice=0.7078 | best=0.7352@37
[stage2_only_256ch] epoch 50/50 | val_dice=0.7212 | best=0.7769@46
[stage2_only_256ch] DONE | best_val_dice=0.7769 (epoch 46) | test_dice=0.6895


### Scenario 5 — MaxSig at Stage 3 only (512 channels)

In [13]:
res = run_scenario(run_name="stage3_only_512ch", maxsig_down_stages=(3,))
ablation_results.append(res)


/tmp/ipykernel_4161971/1748191668.py:14: RuntimeWarning: invalid value encountered in divide
  mask = (mask - mask.min()) / (mask.max() - mask.min())


[stage3_only_512ch] epoch 10/50 | val_dice=0.6693 | best=0.7206@7
[stage3_only_512ch] epoch 20/50 | val_dice=0.6986 | best=0.7257@19
[stage3_only_512ch] epoch 30/50 | val_dice=0.6932 | best=0.7430@26
[stage3_only_512ch] epoch 40/50 | val_dice=0.6436 | best=0.7544@34
[stage3_only_512ch] epoch 50/50 | val_dice=0.7142 | best=0.7544@34
[stage3_only_512ch] DONE | best_val_dice=0.7544 (epoch 34) | test_dice=0.6995


### Scenario 6 — Default configuration: Stages (1, 2) — 128ch + 256ch

In [14]:
res = run_scenario(run_name="default_stages_1_2", maxsig_down_stages=(1, 2))
ablation_results.append(res)
default_result = res  # keep a handle for the GFLOPs/FPS cell below


/tmp/ipykernel_4161971/1748191668.py:14: RuntimeWarning: invalid value encountered in divide
  mask = (mask - mask.min()) / (mask.max() - mask.min())


[default_stages_1_2] epoch 20/50 | val_dice=0.7162 | best=0.7169@14
[default_stages_1_2] epoch 30/50 | val_dice=0.7214 | best=0.7536@21
[default_stages_1_2] epoch 40/50 | val_dice=0.7297 | best=0.7536@21
[default_stages_1_2] epoch 50/50 | val_dice=0.7464 | best=0.7614@44
[default_stages_1_2] DONE | best_val_dice=0.7614 (epoch 44) | test_dice=0.7444


### Scenario 7 — MaxSig at all stages (0, 1, 2, 3)

In [15]:
res = run_scenario(run_name="all_stages_0_1_2_3", maxsig_down_stages=(0, 1, 2, 3))
ablation_results.append(res)


/tmp/ipykernel_4161971/1748191668.py:14: RuntimeWarning: invalid value encountered in divide
  mask = (mask - mask.min()) / (mask.max() - mask.min())


[all_stages_0_1_2_3] epoch 10/50 | val_dice=0.6859 | best=0.7203@8
[all_stages_0_1_2_3] epoch 20/50 | val_dice=0.7376 | best=0.7376@20
[all_stages_0_1_2_3] epoch 30/50 | val_dice=0.7119 | best=0.7436@29
[all_stages_0_1_2_3] epoch 40/50 | val_dice=0.7470 | best=0.7547@35
[all_stages_0_1_2_3] epoch 50/50 | val_dice=0.7511 | best=0.7547@35
[all_stages_0_1_2_3] DONE | best_val_dice=0.7547 (epoch 35) | test_dice=0.7258


## GFLOPs / FPS — default configuration only\n(Stages (1,2), as used in `UNet-maxsig.py`)

In [16]:
default_model = UNet(
    in_channels=1, out_channels=1, features=(64, 128, 256, 512),
    maxsig_down_stages=(1, 2), maxsig_kernel=3, maxsig_mode="gated",
    gamma_init=0.0, use_sigmoid_input=True,
).to(device)

# Load the trained default weights so FPS reflects the actual deployed model
default_model.load_state_dict(torch.load(os.path.join(RESULTS_DIR, "best-default_stages_1_2.pt")))

gflops, fps = compute_flops_fps(default_model, input_size=(1, 1, 256, 256))
print(f"Default config (stages=(1,2)) | GFLOPs: {gflops if gflops is None else round(gflops, 3)} | FPS: {fps:.2f}")


Default config (stages=(1,2)) | GFLOPs: 97.539 | FPS: 26.26


## Summary table

In [17]:
df = pd.DataFrame(ablation_results)
df = df[["run_name", "maxsig_down_stages", "best_val_dice", "best_epoch", "test_dice"]]
df.loc[df["run_name"] == "default_stages_1_2", "GFLOPs"] = gflops
df.loc[df["run_name"] == "default_stages_1_2", "FPS"] = fps
df


,run_name,maxsig_down_stages,best_val_dice,best_epoch,test_dice,GFLOPs,FPS
0,baseline_no_maxsig,(),0.752490,38,0.698194,NaN,NaN
1,stage0_only_64ch,"(0,)",0.746762,31,0.723061,NaN,NaN
2,stage1_only_128ch,"(1,)",0.753885,34,0.733005,NaN,NaN
3,stage2_only_256ch,"(2,)",0.776923,46,0.689493,NaN,NaN
4,stage3_only_512ch,"(3,)",0.754437,34,0.699485,NaN,NaN
5,default_stages_1_2,"(1, 2)",0.761431,44,0.744415,97.53854,26.255102
6,all_stages_0_1_2_3,"(0, 1, 2, 3)",0.754681,35,0.725773,NaN,NaN


In [8]:
baseline_model = UNet(
    in_channels=1, out_channels=1, features=(64, 128, 256, 512),
    maxsig_down_stages=(), maxsig_kernel=3, maxsig_mode="gated",
    gamma_init=0.0, use_sigmoid_input=True,
).to(device)

# Load the trained default weights so FPS reflects the actual deployed model
baseline_model.load_state_dict(torch.load(os.path.join(RESULTS_DIR, "best-baseline_no_maxsig.pt")))

gflops, fps = compute_flops_fps(baseline_model, input_size=(1, 1, 256, 256))
print(f"baseline model | GFLOPs: {gflops if gflops is None else round(gflops, 3)} | FPS: {fps:.2f}")


baseline model | GFLOPs: 96.44 | FPS: 194.99
